# Bentopy Tutorial 3 — Multi-Compartment Systems

This notebook follows the [Bentopy tutorial](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
section *Tutorial 3: Multi-Compartment Systems with Placement Rules*.

We use a **double POPC membrane**, which creates two distinct open
compartments **A** and **B** (separated by the periodic membrane bilayers).
We place

* 200 lysozymes only in compartment **A**, and
* 100 ubiquitins only in compartment **B** *and* close to the membrane.

This combines several earlier ideas: multiple masks, named compartments,
boolean combinations, and proximity rules — all driven from a single
`.bent` recipe.


## 0. Verify the workspace

This notebook is meant to be opened from
`/projects/bgvl/$USER/Martini/tutorial_3/` (created earlier by
`setup_martini.sh`).  All outputs of the cells below land in that folder.


In [ ]:
import os, pathlib, shutil, sys

# Make the bentopy CLI tools available no matter which Python kernel the
# user picks. The shared venv lives in the workshop's allocation.
VENV_BIN = "/projects/bgvl/alfiaparvez/bentopy_tutorial/.venv/bin"
if VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = VENV_BIN + ":" + os.environ["PATH"]

# This notebook is meant to be opened from inside its tutorial folder,
# e.g.  /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_3/ .  Outputs of
# every cell below (.bent, .json, .gro, .top, ...) land in this folder.
NB_DIR = pathlib.Path.cwd()

missing = [d for d in ("structures", "topology", "mdp_files")
           if not (NB_DIR / d).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing input directories in {NB_DIR}: {missing}.\n"
        f"Run  bash Martini/copy.sh  first, then open the notebook from "
        f"/projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_3/."
    )

print("Working dir :", NB_DIR)
print("bentopy-pack:", shutil.which("bentopy-pack"))


## 1. Inspect the compartments of the double membrane

The containment algorithm should now find **four** compartments: two
inter-membrane spaces (`-1`, `-2`) and the two bilayers (`1`, `2`).


In [ ]:
!bentopy-mask structures/double_membrane.gro -b compartment_labels.gro


Useful VMD selections for `compartment_labels.gro`:

* Compartment **A**: `name "-1"`
* Compartment **B**: `name "-2"`
* Membranes (excluded): `name 1 2`


## 2. Create one mask per compartment


In [ ]:
!bentopy-mask structures/double_membrane.gro \
    -l  -1:A_mask.npz                   \
    -l  -2:B_mask.npz                   \
    -l 1,2:membrane_mask.npz


## 3. Write the recipe `compartment_packing.bent`

Note the boolean combination

```
B-close-to-membrane combines membrane-neighborhood and B
```

which restricts ubiquitins to the *intersection* of compartment B and the
4-nm shell around the membranes.


In [ ]:
bent = '''[ general ]
title "Proteins in different compartments"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane                from "membrane_mask.npz"
A                       from "A_mask.npz"
B                       from "B_mask.npz"
membrane-neighborhood   around 4 of membrane
B-close-to-membrane     combines membrane-neighborhood and B

[ segments ]
LYZ:lyz 200 from "structures/lysozyme.pdb"  in A
UBQ:ubq 100 from "structures/ubiquitin.pdb" in B-close-to-membrane
'''
pathlib.Path("compartment_packing.bent").write_text(bent)
print(bent)


## 4. Pack, render, merge


In [ ]:
!bentopy-pack compartment_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top
!bentopy-merge packed_proteins.gro structures/double_membrane.gro -o system.gro
with open('topol.top', 'a') as fh:
    fh.write('POPC    10816\n')


## 5. Solvation


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral
print('--- final topol.top ---')
print(pathlib.Path('topol.top').read_text())


## 5. Visualize

Open the solvated system with VMD (run this in a Delta OnDemand desktop
session, not in the notebook itself):

```bash
export PATH=/projects/bgvl/alfiaparvez/software/vmd/bin:$PATH
cd /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_3
vmd -e vis_t3.tcl solvated_system.gro
```

The QuickSurf rendering script `vis_t3.tcl` is in this folder already,
so VMD will pick it up directly.


### Expected result (Figure 6 of the tutorial)

* Two POPC bilayers slice the box into compartments **A** and **B**.
* Lysozymes (`resname lyz`) live exclusively in compartment **A**.
* Ubiquitins (`resname ubq`) live exclusively in compartment **B**, hugging
  the membrane surfaces (within 4 nm).
* No protein overlaps with either membrane.


## 6. (Optional) Run a GROMACS simulation

The Bentopy tutorial finishes with energy minimisation, an equilibration
and a 1 µs production run on the system you just built.  On Delta this
should be submitted through SLURM, **not** run inside the notebook.

`setup_martini.sh` already placed `run_md.slurm` next to this notebook.
Submit it from a terminal:

```bash
cd /projects/bgvl/$USER/Martini/tutorial_3
sbatch run_md.slurm
squeue -u $USER
```

The template targets the `gpuH200x8` partition under the
`bgvl-delta-gpu` account.  Edit the `#SBATCH` headers if you'd rather
use `gpuA100x4` or `gpuA40x4` (usually shorter queues).


In [ ]:
# Quick sanity check: confirm the SLURM template is in place.
slurm = NB_DIR / 'run_md.slurm'
if slurm.exists():
    print(f'OK: {slurm} ({slurm.stat().st_size} bytes)')
else:
    print(f'MISSING: {slurm}')
    print('Re-run setup_martini.sh to copy it in.')
